# Metodología del Notebook 4 - Análisis de sentimiento de reseñas

## Objetivo

El objetivo del cuarto notebook fue incorporar una dimensión cualitativa al análisis de satisfacción del cliente mediante el estudio del contenido textual de las reseñas. Mientras que la variable `review_score` proporciona una valoración numérica explícita, el texto permite captar el tono de la experiencia expresada por el cliente y enriquecer la interpretación del comportamiento del negocio.

## Procedimiento

En primer lugar, se partió de la tabla maestra generada en el notebook anterior y se seleccionaron únicamente las columnas relevantes para el análisis de reseñas. Posteriormente, se identificaron los pedidos que contaban con comentario textual y se filtró el conjunto para trabajar únicamente con reseñas que contenían mensaje escrito.

Sobre estos comentarios se aplicó una limpieza mínima, limitada a la normalización básica de espacios, con el objetivo de mantener el texto lo más fiel posible al contenido original. A continuación, se utilizó un modelo preentrenado de Hugging Face (`nlptown/bert-base-multilingual-uncased-sentiment`), cargado mediante la librería `Transformers`, para clasificar el sentimiento de cada reseña textual.

El modelo devolvió una predicción de sentimiento que posteriormente fue simplificada en tres clases de análisis: `Negativo`, `Neutro` y `Positivo`. Estas etiquetas fueron integradas de nuevo en la base maestra para poder relacionarlas con otras variables del negocio.

Como parte del control metodológico, el resultado del modelo se validó indirectamente comparando la clasificación de sentimiento con la variable `review_score`. Después, se analizaron las relaciones entre sentimiento y retraso logístico, así como la distribución del sentimiento por categoría principal de producto y por estado geográfico.

## Justificación metodológica

El uso de un modelo preentrenado de Hugging Face permitió incorporar técnicas de procesamiento de lenguaje natural sin necesidad de desarrollar un modelo propio desde cero, lo que resulta apropiado en un proyecto de análisis de negocio aplicado. No obstante, dado que se trata de un modelo generalista y no entrenado específicamente para este dataset, sus resultados no fueron aceptados de forma automática, sino contrastados con variables ya conocidas del negocio para comprobar su coherencia.

## Resultado esperado

El resultado de este notebook fue una clasificación del sentimiento de las reseñas textuales, validada frente a la puntuación numérica y conectada con variables de negocio relevantes. Esta fase permitió demostrar que los pedidos tardíos y determinadas categorías o territorios se asocian con una mayor proporción de sentimiento negativo, reforzando así los hallazgos obtenidos previamente en el análisis exploratorio.

In [63]:
pip install transformers torch

Note: you may need to restart the kernel to use updated packages.


In [64]:
import plotly.express as px
import plotly.graph_objects as go
px.defaults.template = "plotly_white"

In [65]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Carga de archivos

In [66]:
master_df = pd.read_csv("../data/final/master_orders_dataset.csv")
master_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,approval_days,carrier_days,delay_days,is_late,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,n_items,n_products,n_sellers,total_price,total_freight,payment_count,payment_value_total,max_installments,review_score,review_creation_date,review_answer_timestamp,review_comment_title,review_comment_message,main_category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.00,0.00,2.00,-8.00,0.00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,"3,149.00",-23.58,-46.59,sao paulo,SP,1.00,1.00,1.00,29.99,8.72,3.00,38.71,1.00,4.00,2017-10-11 00:00:00,2017-10-12 03:43:48,NaN,"Não testei o produto ainda, mas ele veio corre...",housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.00,1.00,1.00,-6.00,0.00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,"47,813.00",-12.18,-44.66,barreiras,BA,1.00,1.00,1.00,118.70,22.76,1.00,141.46,1.00,4.00,2018-08-08 00:00:00,2018-08-08 18:37:50,Muito boa a loja,Muito bom o produto.,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.00,0.00,0.00,-18.00,0.00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,"75,265.00",-16.75,-48.51,vianopolis,GO,1.00,1.00,1.00,159.90,19.22,1.00,179.12,3.00,5.00,2018-08-18 00:00:00,2018-08-22 19:07:58,NaN,NaN,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.00,0.00,3.00,-13.00,0.00,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,"59,296.00",-5.77,-35.27,sao goncalo do amarante,RN,1.00,1.00,1.00,45.00,27.20,1.00,72.20,1.00,5.00,2017-12-03 00:00:00,2017-12-05 19:21:58,NaN,O produto foi exatamente o que eu esperava e e...,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.00,0.00,0.00,-10.00,0.00,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,"9,195.00",-23.68,-46.51,santo andre,SP,1.00,1.00,1.00,19.90,8.72,1.00,28.62,1.00,5.00,2018-02-17 00:00:00,2018-02-18 13:02:51,NaN,NaN,stationery


## Aseguramos la carga de fechas en el formato correcto:

In [67]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "review_creation_date",
    "review_answer_timestamp"
]

for col in date_cols:
    if col in master_df.columns:
        master_df[col] = pd.to_datetime(master_df[col], errors="coerce")

# Selección de columnas utiles para nuestro analisis:

In [68]:
reviews_df = master_df[[
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "delay_days",
    "is_late",
    "delivery_days",
    "main_category",
    "geolocation_state"
]].copy()

reviews_df.head()

,order_id,review_score,review_comment_title,review_comment_message,delay_days,is_late,delivery_days,main_category,geolocation_state
0,e481f51cbdc54678b7cc49136f2d6af7,4.00,NaN,"Não testei o produto ainda, mas ele veio corre...",-8.00,0.00,8.00,housewares,SP
1,53cdb2fc8bc7dce0b6741e2150273451,4.00,Muito boa a loja,Muito bom o produto.,-6.00,0.00,13.00,perfumery,BA
2,47770eb9100c2d0c44946d9cf07ec65d,5.00,NaN,NaN,-18.00,0.00,9.00,auto,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,5.00,NaN,O produto foi exatamente o que eu esperava e e...,-13.00,0.00,13.00,pet_shop,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,5.00,NaN,NaN,-10.00,0.00,2.00,stationery,SP


## Vemos primero cuantas reseñas contienen texto real:

In [69]:
reviews_df["has_text"] = (
    reviews_df["review_comment_message"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

reviews_df["has_text"].value_counts()

has_text
False    58632
True     40809
Name: count, dtype: int64

## Disponibilidad de texto en las reseñas

En este paso se identifica cuántas observaciones contienen un comentario escrito por el cliente. Esta comprobación es necesaria porque el análisis de sentimiento solo puede aplicarse sobre texto real. Las reseñas sin comentario textual no se utilizarán en esta fase, aunque seguirán siendo útiles en otros análisis a través de la variable `review_score`.

## Filtrado de reseñas con texto:

In [70]:
reviews_text_df = reviews_df[reviews_df["has_text"]].copy()
reviews_text_df.shape

(40809, 10)

In [71]:
def clean_review_text(text):
    text = str(text).strip()
    text = " ".join(text.split())
    return text

reviews_text_df["review_text_clean"] = reviews_text_df["review_comment_message"].apply(clean_review_text)

reviews_text_df[["review_comment_message", "review_text_clean"]].head()

,review_comment_message,review_text_clean
0,"Não testei o produto ainda, mas ele veio corre...","Não testei o produto ainda, mas ele veio corre..."
1,Muito bom o produto.,Muito bom o produto.
3,O produto foi exatamente o que eu esperava e e...,O produto foi exatamente o que eu esperava e e...
6,fiquei triste por n ter me atendido.,fiquei triste por n ter me atendido.
10,Aguardando retorno da loja,Aguardando retorno da loja


## Preparación del texto

La limpieza aplicada al texto es deliberadamente simple. En lugar de transformar agresivamente las reseñas, únicamente se eliminan espacios sobrantes al inicio, al final y entre palabras. De este modo, el contenido original del comentario se conserva prácticamente intacto y se facilita su procesamiento por parte del modelo de sentimiento.

## Cargamos un modelo de sentimiento multilingüe para una mejor interpretación de los resultados:

In [72]:
model_name = "nlptown/bert-base-multilingual-uncased-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

sentiment_map_5 = {
    0: "Muy negativo",
    1: "Negativo",
    2: "Neutro",
    3: "Positivo",
    4: "Muy positivo"
}

print("Modelo cargado en:", device)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo cargado en: cpu


## Hacemos prueba del modelo preentrenado:

In [73]:
sample_texts = reviews_text_df["review_text_clean"].head(5).tolist()

inputs = tokenizer(
    sample_texts,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=256
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    preds = torch.argmax(probs, dim=-1).cpu().numpy()

for text, pred in zip(sample_texts, preds):
    print("Texto:", text[:150], "...")
    print("Predicción:", sentiment_map_5[int(pred)])
    print("-" * 80)

Texto: Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se t ...
Predicción: Neutro
--------------------------------------------------------------------------------
Texto: Muito bom o produto. ...
Predicción: Muy positivo
--------------------------------------------------------------------------------
Texto: O produto foi exatamente o que eu esperava e estava descrito no site e chegou bem antes da data prevista. ...
Predicción: Muy positivo
--------------------------------------------------------------------------------
Texto: fiquei triste por n ter me atendido. ...
Predicción: Muy negativo
--------------------------------------------------------------------------------
Texto: Aguardando retorno da loja ...
Predicción: Positivo
--------------------------------------------------------------------------------


## Creamos una función para predecir por lotes:

In [74]:
def predict_sentiment_batch(texts, batch_size=32, max_length=256):
    predictions = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            pred_ids = torch.argmax(probs, dim=-1).cpu().numpy()
            confidences = probs.max(dim=-1).values.cpu().numpy()

        for pred_id, conf in zip(pred_ids, confidences):
            predictions.append((sentiment_map_5[int(pred_id)], float(conf)))

    return predictions

## Aplicamos la predicción sobre todas las reseñas con texto:

In [75]:
predictions = predict_sentiment_batch(
    reviews_text_df["review_text_clean"].tolist(),
    batch_size=32,
    max_length=256
)

reviews_text_df["sentiment_5"] = [p[0] for p in predictions]
reviews_text_df["sentiment_confidence"] = [p[1] for p in predictions]

reviews_text_df[[
    "review_text_clean",
    "review_score",
    "sentiment_5",
    "sentiment_confidence"
]].head(10)

,review_text_clean,review_score,sentiment_5,sentiment_confidence
0,"Não testei o produto ainda, mas ele veio corre...",4.00,Neutro,0.53
1,Muito bom o produto.,4.00,Muy positivo,0.56
3,O produto foi exatamente o que eu esperava e e...,5.00,Muy positivo,0.46
6,fiquei triste por n ter me atendido.,2.00,Muy negativo,0.42
10,Aguardando retorno da loja,1.00,Positivo,0.36
14,Gostei do produto,4.00,Muy positivo,0.47
15,Obrigado pela atenção. Lojas lannister perfeit...,5.00,Muy positivo,0.68
20,os correios estäo em greve... näo recebi nenhu...,2.00,Muy negativo,0.61
24,Sempre vou comprar aqui pois é a melhor parabéns,5.00,Muy positivo,0.73
25,Demora muito entregar. Já passou o prazo e ain...,2.00,Muy negativo,0.52


## Simplificación de las clases de sentimiento:

Aunque el modelo devuelve cinco niveles de sentimiento, para el análisis del negocio resulta más útil trabajar con tres categorías más simples: `Negativo`, `Neutro` y `Positivo`. Esta reducción facilita la interpretación de resultados y hace más clara la presentación posterior en tablas, gráficos y dashboards.

In [76]:
def collapse_sentiment(label):
    if label in ["Muy negativo", "Negativo"]:
        return "Negativo"
    elif label == "Neutro":
        return "Neutro"
    else:
        return "Positivo"

reviews_text_df["sentiment_3"] = reviews_text_df["sentiment_5"].apply(collapse_sentiment)

reviews_text_df["sentiment_3"].value_counts()

sentiment_3
Positivo    21615
Negativo    15484
Neutro       3710
Name: count, dtype: int64

## Presentación de la distribución de sentimiento:

In [77]:
sentiment_distribution = (
    reviews_text_df["sentiment_3"]
    .value_counts()
    .rename_axis("Sentimiento")
    .reset_index(name="Cantidad")
)

sentiment_distribution["Porcentaje"] = (
    sentiment_distribution["Cantidad"] / sentiment_distribution["Cantidad"].sum() * 100
).round(2)

sentiment_distribution

,Sentimiento,Cantidad,Porcentaje
0,Positivo,21615,52.97
1,Negativo,15484,37.94
2,Neutro,3710,9.09


In [78]:
fig = px.pie(
    sentiment_distribution,
    names="Sentimiento",
    values="Cantidad",
    hole=0.45,
    title="Distribución general del sentimiento en las reseñas"
)

fig.show()

## Interpretación de la distribución del sentimiento

Esta tabla y su gráfico asociado permiten observar cómo se reparte el sentimiento general de las reseñas textuales del ecommerce. A través de esta distribución se obtiene una primera aproximación a la percepción global de los clientes, diferenciando entre comentarios positivos, neutros y negativos.

La clasificación de las reseñas textuales muestra que el sentimiento predominante es positivo, lo que sugiere que, en términos generales, una parte importante de los clientes expresa una experiencia favorable en sus comentarios. Sin embargo, la proporción de reseñas negativas sigue siendo elevada, lo que indica que existe un volumen relevante de clientes insatisfechos cuya experiencia no puede considerarse marginal.

Este resultado aporta una visión complementaria a la puntuación numérica de las reseñas, ya que permite captar de forma más rica el tono general del comentario y no solo la valoración final asignada por el cliente.

In [79]:
sentiment_cols = reviews_text_df[[
    "order_id",
    "sentiment_5",
    "sentiment_3",
    "sentiment_confidence"
]].copy()

master_df = master_df.merge(sentiment_cols, on="order_id", how="left")

master_df[[
    "order_id",
    "review_score",
    "review_comment_message",
    "sentiment_3",
    "sentiment_confidence"
]].head()

,order_id,review_score,review_comment_message,sentiment_3,sentiment_confidence
0,e481f51cbdc54678b7cc49136f2d6af7,4.00,"Não testei o produto ainda, mas ele veio corre...",Neutro,0.53
1,53cdb2fc8bc7dce0b6741e2150273451,4.00,Muito bom o produto.,Positivo,0.56
2,47770eb9100c2d0c44946d9cf07ec65d,5.00,NaN,NaN,NaN
3,949d5b44dbf5de918fe9c16f97b45f8a,5.00,O produto foi exatamente o que eu esperava e e...,Positivo,0.46
4,ad21c59c0840e6cb83a9ceb5573f8159,5.00,NaN,NaN,NaN


## Validamos resultado con el `review_score`

In [80]:
sentiment_vs_score = pd.crosstab(
    reviews_text_df["review_score"],
    reviews_text_df["sentiment_3"],
    normalize="index"
).round(3) * 100

sentiment_vs_score

sentiment_3,Negativo,Neutro,Positivo
review_score,,,
1.00,91.60,3.70,4.80
1.50,100.00,0.00,0.00
2.00,81.40,11.20,7.40
2.50,80.00,12.00,8.00
3.00,56.20,21.40,22.40
3.33,100.00,0.00,0.00
3.50,60.00,33.30,6.70
4.00,22.70,16.80,60.50
4.50,46.20,30.80,23.10


## Validación frente a la puntuación numérica de la reseña

Para comprobar si el modelo de sentimiento está produciendo resultados razonables, se compara la clase de sentimiento predicha con la variable `review_score`. En términos generales, se espera que las puntuaciones bajas presenten una mayor proporción de sentimiento negativo, mientras que las puntuaciones altas deberían concentrar más sentimiento positivo.

Esta comparación no constituye una validación perfecta, pero sí ofrece una comprobación práctica muy útil para determinar si la salida del modelo es coherente con la valoración explícita realizada por el cliente.

La comparación entre el sentimiento predicho y la puntuación numérica de la reseña muestra una relación coherente entre ambas variables. En general, las valoraciones más bajas concentran una proporción claramente superior de sentimiento negativo, mientras que las valoraciones más altas se asocian con una presencia mucho mayor de sentimiento positivo.

Este comportamiento sugiere que el modelo de sentimiento está captando de forma razonable la orientación general de las opiniones de los clientes. Por tanto, los resultados pueden considerarse suficientemente consistentes como para incorporarlos al análisis del negocio y utilizarlos en cruces posteriores con variables logísticas, geográficas y de categoría de producto.

In [81]:
sentiment_vs_score_plot = sentiment_vs_score.reset_index()

fig = px.bar(
    sentiment_vs_score_plot,
    x="review_score",
    y=["Negativo", "Neutro", "Positivo"],
    title="Distribución del sentimiento según review score",
    barmode="stack"
)

fig.update_layout(
    xaxis_title="Review score",
    yaxis_title="Porcentaje"
)

fig.show()

## Valores intermedios de review score

En algunos casos aparecen valores intermedios de `review_score`, como 1.5, 2.5, 3.33 o 4.5. Esto se debe a que, durante la preparación de la base analítica, la variable de puntuación se consolidó a nivel de pedido mediante una agregación promedio cuando existían múltiples registros asociados al mismo pedido.

Este detalle no invalida la interpretación general del análisis, pero conviene tenerlo en cuenta al leer la relación entre puntuación y sentimiento.

## Cruzamos sentimientos con atrasos:

## Relación entre sentimiento y retraso logístico

En esta fase se analiza si el sentimiento expresado en las reseñas cambia según el pedido haya sido entregado a tiempo o con retraso. El objetivo es comprobar si los problemas logísticos detectados en el análisis exploratorio anterior también se reflejan en el contenido textual de los comentarios de los clientes.

Si los pedidos tardíos muestran una mayor proporción de reseñas negativas, ello reforzará la hipótesis de que la logística influye de forma directa en la experiencia del cliente.

In [82]:
sentiment_vs_late = (
    reviews_text_df
    .groupby("is_late", as_index=False)["sentiment_3"]
    .value_counts(normalize=True)
)

sentiment_vs_late["proportion"] = (sentiment_vs_late["proportion"] * 100).round(2)

sentiment_vs_late

,is_late,sentiment_3,proportion
0,0.00,Positivo,59.15
1,0.00,Negativo,31.27
2,0.00,Neutro,9.58
3,1.00,Negativo,76.61
4,1.00,Positivo,16.58
5,1.00,Neutro,6.80


## Grafico de Sentimiento vs Atraso:

In [83]:
pivot_late = sentiment_vs_late.pivot(index="is_late", columns="sentiment_3", values="proportion").reset_index()

fig = px.bar(
    pivot_late,
    x="is_late",
    y=["Negativo", "Neutro", "Positivo"],
    title="Sentimiento según estado de entrega",
    barmode="stack"
)

fig.update_layout(
    xaxis_title="Estado de entrega",
    yaxis_title="Porcentaje"
)

fig.show()

## Interpretación de la relación entre sentimiento y retraso logístico

El cruce entre sentimiento y estado de entrega muestra una diferencia muy marcada entre pedidos entregados a tiempo y pedidos tardíos. En los pedidos que llegan dentro del plazo previsto predomina claramente el sentimiento positivo, mientras que en los pedidos con retraso el sentimiento negativo pasa a ser ampliamente mayoritario.

Este resultado constituye una de las evidencias más sólidas del análisis, ya que sugiere una asociación muy fuerte entre el desempeño logístico y la experiencia del cliente expresada en texto. En otras palabras, los retrasos no solo afectan al cumplimiento operativo, sino también a la percepción general del servicio y al tono de las reseñas.

## Sentimientos por Categorias:

In [84]:
reviews_text_df["main_category"] = reviews_text_df["main_category"].fillna("Unknown")

sentiment_by_category = (
    reviews_text_df
    .groupby("main_category", as_index=False)["sentiment_3"]
    .value_counts(normalize=True)
)

sentiment_by_category["proportion"] = (sentiment_by_category["proportion"] * 100).round(2)

sentiment_by_category.head(20)

,main_category,sentiment_3,proportion
0,Unknown,Negativo,62.80
1,Unknown,Positivo,29.88
2,Unknown,Neutro,7.31
3,agro_industry_and_commerce,Positivo,47.46
4,agro_industry_and_commerce,Negativo,44.07
5,agro_industry_and_commerce,Neutro,8.47
6,air_conditioning,Negativo,42.72
7,air_conditioning,Positivo,42.72
8,air_conditioning,Neutro,14.56
9,art,Positivo,46.51


## Filtramos categorias con mucho mas peso:

In [85]:
category_counts_text = (
    reviews_text_df["main_category"]
    .value_counts()
    .reset_index()
)

category_counts_text.columns = ["main_category", "n_reviews_text"]

top_categories_text = category_counts_text.head(10)["main_category"].tolist()

sentiment_top_categories = sentiment_by_category[
    sentiment_by_category["main_category"].isin(top_categories_text)
].copy()

sentiment_top_categories.head(20)

,main_category,sentiment_3,proportion
18,auto,Positivo,55.02
19,auto,Negativo,36.19
20,auto,Neutro,8.79
24,bed_bath_table,Positivo,51.57
25,bed_bath_table,Negativo,38.42
26,bed_bath_table,Neutro,10.01
46,computers_accessories,Positivo,49.17
47,computers_accessories,Negativo,41.55
48,computers_accessories,Neutro,9.28
115,furniture_decor,Positivo,49.66


### Tabla pivote de sentimiento por categoria:

In [86]:
pivot_category_sentiment = sentiment_top_categories.pivot(
    index="main_category",
    columns="sentiment_3",
    values="proportion"
).fillna(0)

pivot_category_sentiment

sentiment_3,Negativo,Neutro,Positivo
main_category,,,
auto,36.19,8.79,55.02
bed_bath_table,38.42,10.01,51.57
computers_accessories,41.55,9.28,49.17
furniture_decor,41.04,9.30,49.66
garden_tools,35.33,7.27,57.40
health_beauty,36.88,8.97,54.15
housewares,36.73,9.76,53.51
sports_leisure,36.13,8.06,55.81
telephony,40.16,9.65,50.19


In [87]:
pivot_category_sentiment_reset = pivot_category_sentiment.reset_index()

fig = px.bar(
    pivot_category_sentiment_reset,
    x="main_category",
    y=["Negativo", "Neutro", "Positivo"],
    title="Distribución del sentimiento en las principales categorías",
    barmode="stack"
)

fig.update_layout(
    xaxis_title="Categoría",
    yaxis_title="Porcentaje",
    xaxis_tickangle=-45
)

fig.show()

# Distribucion de sentimientos por Estados:

In [88]:
sentiment_by_state = (
    reviews_text_df
    .groupby("geolocation_state", as_index=False)["sentiment_3"]
    .value_counts(normalize=True)
)

sentiment_by_state["proportion"] = (sentiment_by_state["proportion"] * 100).round(2)

sentiment_by_state.head(20)

,geolocation_state,sentiment_3,proportion
0,AC,Positivo,50.00
1,AC,Negativo,44.74
2,AC,Neutro,5.26
3,AL,Negativo,51.50
4,AL,Positivo,39.00
5,AL,Neutro,9.50
6,AM,Positivo,61.90
7,AM,Negativo,31.75
8,AM,Neutro,6.35
9,AP,Positivo,70.00


## Filtramos los Estados mas relevantes:

In [89]:
state_counts_text = (
    reviews_text_df["geolocation_state"]
    .value_counts()
    .reset_index()
)

state_counts_text.columns = ["geolocation_state", "n_reviews_text"]

top_states_text = state_counts_text.head(10)["geolocation_state"].tolist()

sentiment_top_states = sentiment_by_state[
    sentiment_by_state["geolocation_state"].isin(top_states_text)
].copy()

pivot_state_sentiment = sentiment_top_states.pivot(
    index="geolocation_state",
    columns="sentiment_3",
    values="proportion"
).fillna(0)

pivot_state_sentiment

sentiment_3,Negativo,Neutro,Positivo
geolocation_state,,,
BA,39.96,9.61,50.42
ES,35.64,11.02,53.34
GO,36.39,9.27,54.34
MG,35.40,8.75,55.84
PE,37.68,8.87,53.45
PR,37.11,8.76,54.14
RJ,44.71,8.51,46.78
RS,38.73,9.06,52.22
SC,39.75,9.04,51.21


In [90]:
pivot_state_sentiment_reset = pivot_state_sentiment.reset_index()

fig = px.bar(
    pivot_state_sentiment_reset,
    x="geolocation_state",
    y=["Negativo", "Neutro", "Positivo"],
    title="Distribución del sentimiento en los principales estados",
    barmode="stack"
)

fig.update_layout(
    xaxis_title="Estado",
    yaxis_title="Porcentaje"
)

fig.show()

In [91]:
reviews_text_df[[
    "review_text_clean",
    "review_score",
    "sentiment_5",
    "sentiment_3",
    "sentiment_confidence"
]].sample(15, random_state=42)

,review_text_clean,review_score,sentiment_5,sentiment_3,sentiment_confidence
38958,Não nada a declarar Não cumpre o que promete,1.00,Muy negativo,Negativo,0.63
1087,"Linda a boneca, carinha querida,amei",5.00,Muy positivo,Positivo,0.52
569,"Produto de ótima qualidade, fiquei super feliz...",5.00,Muy positivo,Positivo,0.76
55111,mt bom o produto,5.00,Positivo,Positivo,0.40
15912,ja comprei algum tempo atraz na shptime e comp...,5.00,Muy negativo,Negativo,0.62
35472,"Só acho que o tempo de entrega é muito longo, ...",1.00,Neutro,Neutro,0.46
24990,Assim que receber posso avaliar,1.00,Muy positivo,Positivo,0.27
27426,celular muito bom gostei bem rápido e super le...,5.00,Muy positivo,Positivo,0.75
44521,Recebi apenas metade dos produtos e não consig...,1.00,Muy negativo,Negativo,0.76
30608,Deu tudo certo!,4.00,Muy positivo,Positivo,0.55


## Revisión manual de ejemplos

Además de la validación cuantitativa, se revisan manualmente algunos ejemplos concretos para comprobar si la clasificación del modelo resulta razonable al comparar el texto, la puntuación de la reseña y la etiqueta de sentimiento asignada. Esta comprobación cualitativa ayuda a identificar posibles errores o casos ambiguos y mejora la confianza en los resultados del análisis.

In [92]:
master_df.to_csv("../data/final/master_orders_with_sentiment.csv", index=False)

# Conclusiones del análisis de sentimiento

El análisis de sentimiento de las reseñas textuales permite complementar la información proporcionada por la puntuación numérica y añadir una dimensión cualitativa a la experiencia del cliente. Los resultados muestran que, aunque el sentimiento positivo es predominante, existe una proporción relevante de comentarios negativos que no puede considerarse residual.

La validación frente al `review_score` sugiere que el modelo utilizado ofrece resultados coherentes con la valoración explícita del cliente. Además, el cruce entre sentimiento y retraso logístico muestra una relación muy clara: los pedidos tardíos concentran una proporción mucho mayor de reseñas negativas que los pedidos entregados a tiempo. Este hallazgo refuerza la idea de que la logística desempeña un papel central en la percepción global del servicio y en la satisfacción del cliente.

En esta fase del proyecto se realizó un análisis de sentimiento sobre las reseñas textuales de los clientes con el objetivo de complementar la información proporcionada por la variable numérica `review_score`. Para ello, se utilizó un modelo preentrenado de Hugging Face, concretamente `nlptown/bert-base-multilingual-uncased-sentiment`, cargado mediante la librería `Transformers`.

La elección de este enfoque permitió trabajar con un modelo ya entrenado para clasificación de sentimiento sin necesidad de desarrollar un proceso de entrenamiento propio, lo que resulta adecuado en un proyecto de análisis aplicado al negocio. A partir de las reseñas con texto disponible, se realizó una limpieza mínima del contenido para mantener la mayor fidelidad posible al comentario original y, posteriormente, se aplicó el modelo para obtener una clasificación automática del sentimiento.

Las salidas originales del modelo se agruparon después en tres categorías más interpretables desde el punto de vista analítico: `Negativo`, `Neutro` y `Positivo`. Finalmente, estas clasificaciones se validaron mediante su comparación con la variable `review_score` y se relacionaron con variables relevantes del negocio, como el retraso logístico, la categoría principal del producto y la localización geográfica del cliente.